# Create annual and temperature based predictions from the best models

In this notebook we'll create predictions for surface type specific annual budget and temperature specific flux using the full model equation and by setting $x_{i,j} = 1$ for each surface type j yielding a out-of-model equation

$$
E(\ln{F_i} | x_{i,j}=1) = (\alpha + \gamma_j) + (\beta + \delta_j) \frac{T_{\mathrm{air,i,j}} - 10^{\circ} \mathrm{C}}{10^{\circ} \mathrm{C}} + \zeta\theta
$$

In [1]:
import cloudpickle
import pymc as pm
import arviz as az
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

%matplotlib widget
%load_ext autoreload
%autoreload 2

WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


# Load data to extract soil class names

In [2]:
data_n2o = pd.read_csv('data/inference_data_n2o.csv', index_col='time')
data_n2o.index = pd.to_datetime(data_n2o.index)

data_ch4 = pd.read_csv('data/inference_data_ch4.csv', index_col='time')
data_ch4.index = pd.to_datetime(data_ch4.index)

# Open the best models

In [3]:
with open('models/full_model_n2o_st9_theta_mutable.pkl','rb') as buff:
    model_n2o = cloudpickle.load(buff)

with open('models/full_model_ch4_st9_theta_mutable.pkl','rb') as buff:
    model_ch4 = cloudpickle.load(buff)

In [ ]:
new_st_9 = np.array(['Dead wood','Harvest residue','Exposed peat','Litter','Bottom layer (mosses)','Field layer','Ditch (water surface)', 'Living tree', 'Plant covered ditch'])

# Load T<sub>air</sub> data for annual budget

In [5]:
Tair = pd.read_csv('data/T_air_gapfilled.csv')
T = Tair['T_air'].values
T = T.reshape(T.shape[0], 1)
T_air_annual_st_9 = np.repeat(T, repeats=9, axis=1)
T_air_annual_st_6 = np.repeat(T, repeats=6, axis=1)

# Load $\theta$ data for annual budget

In [6]:
theta = pd.read_csv('data/theta_snow_free_period.csv', index_col='time')

In [7]:
theta_annual = theta.loc[:, 'soil_moisture_tomst_mean'].values

# Define surface type fractions

In [8]:
# Edit the copy pasted values (the last one is the instrument class which we don't want)

X_st_9 = np.array([0.22831063575874028,
 0.07912675405730267,
 0.29005355713693703,
 0.19903329159584093,
 0.014206532335570385,
 0.11784682410999092,
 0.018604968072915225,
 0.04190943651265434,
 0.010801791670784143,
])

X_st_9 = X_st_9.reshape(1, -1)

# N<sub>2</sub>O predictions

In [10]:
ind = ((data_n2o.T_air <= 11) & (data_n2o.T_air >= 9))
n2o_tenC_mean, n2o_tenC_std = ((data_n2o.loc[ind, 'F_N2O_ln']).mean(), (data_n2o.loc[ind, 'F_N2O_ln']).std())

## Annual T<sub>air</sub> for the whole plot

In [11]:
gamma_std = 2.0
lambda_beta = 1.0
lambda_delta = 1.0
lambda_epsilon = 1.0
alpha_mu_mean, alpha_mu_std = n2o_tenC_mean, n2o_tenC_std

coords = {'st': new_st_9, 'T_id': np.arange(T_air_annual_st_9.shape[0])}
with pm.Model(coords=coords) as new_model:
    T_air = T_air_annual_st_9
    theta = theta_annual
    st_frac = np.repeat(X_st_9, repeats=T_air_annual_st_9.shape[0], axis=0)
    alpha = pm.Normal("alpha", mu=alpha_mu_mean, sigma=alpha_mu_std)
    beta = pm.Exponential('beta', lambda_beta)

    gamma_mu = pm.Normal("gamma_mu", mu=0, sigma=1)
    gamma = pm.Normal("gamma", mu=gamma_mu, sigma=gamma_std, dims='st') 
    zeta = pm.Normal('zeta', mu=0, sigma=2)
    delta = pm.Exponential("delta", lambda_delta, dims='st')

    epsilon = pm.Exponential("sigma", lambda_epsilon)

    pred = pm.Normal("pred", mu = alpha + beta*(T_air[:, 0]-10)/10 + zeta*theta + pm.math.sum(gamma*st_frac + delta*(T_air-10)/10*st_frac, axis=1), sigma=epsilon)

    annual_n2o = pm.sample_posterior_predictive(model_n2o['model_res'].posterior, var_names=["pred"], predictions=True)

Sampling: [pred]


In [12]:
annual_n2o.predictions['sigma'] = (["chain", "draw"], model_n2o['model_res'].posterior.sigma.data)

In [13]:
annual_n2o.predictions['pred_real_bias_corrected'] = (["chain", "draw", "pred_dim_2"], (np.exp(annual_n2o.predictions.pred.data + annual_n2o.predictions.sigma.data[:, :, None]**2.0/2.0)))

### Rename dims

In [14]:
dates = pd.date_range(datetime(2022,5,1), datetime(2022,11,16), freq="30min")

In [15]:
annual_n2o = annual_n2o.predictions
annual_n2o = annual_n2o.rename_dims({'pred_dim_2': 'time'})
annual_n2o = annual_n2o.rename_vars({'pred_dim_2': 'time'})
annual_n2o['time'] = dates

### Save

In [16]:
annual_n2o.to_netcdf('data/n2o_pred_annual_T_air_theta.nc')

## Annual T for each surface type

In [30]:
gamma_std = 2.0
lambda_beta = 1.0
lambda_delta = 1.0
lambda_epsilon = 1.0
alpha_mu_mean, alpha_mu_std = n2o_tenC_mean, n2o_tenC_std

coords = {'st': new_st_9, 'T_id': np.arange(T_air_annual_st_9.shape[0])}
with pm.Model(coords=coords) as new_model:
    T_air = T_air_annual_st_9
    theta = theta_annual.reshape((theta_annual.shape[0],1))
    alpha = pm.Normal("alpha", mu=alpha_mu_mean, sigma=alpha_mu_std)
    beta = pm.Exponential('beta', lambda_beta)
    zeta = pm.Normal('zeta', mu=0, sigma=2)

    gamma_mu = pm.Normal("gamma_mu", mu=0, sigma=1)
    gamma = pm.Normal("gamma", mu=gamma_mu, sigma=gamma_std, dims='st') 

    delta = pm.Exponential("delta", lambda_delta, dims='st')

    epsilon = pm.Exponential("sigma", lambda_epsilon)

    pred = pm.Normal("pred", mu = (alpha+gamma) + (beta+delta)*(T_air-10)/10 + zeta*theta, sigma=epsilon)

    annual_st_n2o = pm.sample_posterior_predictive(model_n2o['model_res'], var_names=["pred"], predictions=True)

Sampling: [pred]


### Add sigma and bias corrected prediction

In [31]:
annual_st_n2o.predictions['sigma'] = (["chain", "draw"], model_n2o['model_res'].posterior.sigma.data)

In [32]:
annual_st_n2o.predictions['pred_real_bias_corrected'] = (["chain", "draw", "pred_dim_2", "pred_dim_3"], (np.exp(annual_st_n2o.predictions.pred.data + annual_st_n2o.predictions.sigma.data[:, :, None, None]**2.0/2.0)))

### Rename dims

In [33]:
annual_st_n2o = annual_st_n2o.predictions
annual_st_n2o = annual_st_n2o.rename_dims({'pred_dim_2': 'time'})
annual_st_n2o = annual_st_n2o.rename_vars({'pred_dim_2': 'time'})
annual_st_n2o['time'] = dates

In [34]:
annual_st_n2o = annual_st_n2o.rename_dims({'pred_dim_3': 'soil_classes'})
annual_st_n2o = annual_st_n2o.rename_vars({'pred_dim_3': 'soil_classes'})
annual_st_n2o['soil_classes'] = new_st_9

In [35]:
annual_st_n2o.to_netcdf('data/annual_st_n2o_revision.nc')

# CH<sub>4</sub> predictions

In [17]:
ind = ((data_ch4.T_air <= 11) & (data_ch4.T_air >= 9))
ch4_tenC_mean, ch4_tenC_std = ((data_ch4.loc[ind, 'F_CH4_ln']).mean(), (data_ch4.loc[ind, 'F_CH4_ln']).std())

## Annual T<sub>air</sub>

In [18]:
gamma_std = 2.0
lambda_beta = 1.0
lambda_delta = 1.0
lambda_epsilon = 1.0
alpha_mu_mean, alpha_mu_std = ch4_tenC_mean, ch4_tenC_std

coords = {'st': new_st_9, 'T_id': np.arange(T_air_annual_st_6.shape[0])}
with pm.Model(coords=coords) as new_model:
    T_air = T_air_annual_st_9
    theta = theta_annual
    st_frac = np.repeat(X_st_9, repeats=T_air_annual_st_6.shape[0], axis=0)
    alpha = pm.Normal("alpha", mu=alpha_mu_mean, sigma=alpha_mu_std)
    beta = pm.Exponential('beta', lambda_beta)
    zeta = pm.Normal('zeta', mu=0, sigma=2)
    gamma_mu = pm.Normal("gamma_mu", mu=0, sigma=1)
    gamma = pm.Normal("gamma", mu=gamma_mu, sigma=gamma_std, dims='st') 

    delta = pm.Exponential("delta", lambda_delta, dims='st')

    epsilon = pm.Exponential("sigma", lambda_epsilon)

    pred = pm.Normal("pred", mu = alpha + beta*(T_air[:, 0]-10)/10 + zeta*theta + pm.math.sum(gamma*st_frac + delta*(T_air-10)/10*st_frac, axis=1), sigma=epsilon)

    annual_ch4 = pm.sample_posterior_predictive(model_ch4['model_res'], var_names=["pred"], predictions=True)

Sampling: [pred]


### Add sigma and bias corrected prediction

In [19]:
annual_ch4.predictions['sigma'] = (["chain", "draw"], model_ch4['model_res'].posterior.sigma.data)

In [20]:
annual_ch4.predictions['pred_real_bias_corrected'] = (["chain", "draw", "pred_dim_2"], (np.exp(annual_ch4.predictions.pred.data + annual_ch4.predictions.sigma.data[:, :, None]**2.0/2.0)-10))

### Rename dims

In [21]:
annual_ch4 = annual_ch4.predictions
annual_ch4 = annual_ch4.rename_dims({'pred_dim_2': 'time'})
annual_ch4 = annual_ch4.rename_vars({'pred_dim_2': 'time'})
annual_ch4['time'] = dates

### Save

In [22]:
annual_ch4.to_netcdf('data/ch4_pred_annual_T_air_theta.nc')

## Annual T for each surface type

In [18]:
gamma_std = 2.0
lambda_beta = 1.0
lambda_delta = 1.0
lambda_epsilon = 1.0
alpha_mu_mean, alpha_mu_std = ch4_tenC_mean, ch4_tenC_std

coords = {'st': new_st_9, 'T_id': np.arange(T_air_annual_st_9.shape[0]), 'theta_id': np.arange(theta_annual.shape[0])}
with pm.Model(coords=coords) as new_model:
    T_air = T_air_annual_st_9
    theta = theta_annual.reshape((theta_annual.shape[0],1))
    alpha = pm.Normal("alpha", mu=alpha_mu_mean, sigma=alpha_mu_std)
    beta = pm.Exponential('beta', lambda_beta)
    zeta = pm.Normal('zeta', mu=0, sigma=2)
    gamma_mu = pm.Normal("gamma_mu", mu=0, sigma=1)
    gamma = pm.Normal("gamma", mu=gamma_mu, sigma=gamma_std, dims='st') 

    delta = pm.Exponential("delta", lambda_delta, dims='st')

    epsilon = pm.Exponential("sigma", lambda_epsilon)

    pred = pm.Normal("pred", mu = (alpha+gamma) + (beta+delta)*(T_air-10)/10 + zeta*theta, sigma=epsilon)

    annual_st_ch4 = pm.sample_posterior_predictive(model_ch4['model_res'], var_names=["pred"], predictions=True)

Sampling: [pred]


### Add sigma and bias corrected prediction

In [19]:
annual_st_ch4.predictions['sigma'] = (["chain", "draw"], model_ch4['model_res'].posterior.sigma.data)

In [20]:
annual_st_ch4.predictions['pred_real_bias_corrected'] = (["chain", "draw", "pred_dim_2", "pred_dim_3"], (np.exp(annual_st_ch4.predictions.pred.data + annual_st_ch4.predictions.sigma.data[:, :, None, None]**2.0/2.0)-10))

### Rename dims

In [21]:
annual_st_ch4 = annual_st_ch4.predictions
annual_st_ch4 = annual_st_ch4.rename_dims({'pred_dim_2': 'time'})
annual_st_ch4 = annual_st_ch4.rename_vars({'pred_dim_2': 'time'})
annual_st_ch4['time'] = dates

In [22]:
annual_st_ch4 = annual_st_ch4.rename_dims({'pred_dim_3': 'soil_classes'})
annual_st_ch4 = annual_st_ch4.rename_vars({'pred_dim_3': 'soil_classes'})
annual_st_ch4['soil_classes'] = new_st_9

In [23]:
annual_st_ch4.to_netcdf('data/annual_st_ch4_revision.nc')